# Optimisation hyperparamètres et seuil de décision d'un Régression Logistique
From Gemini

## Recherche des hyperparamètres par GrilleSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Créer un jeu de données synthétique
X, y = make_classification(n_samples=1000, n_features=20, n_informative=2, n_redundant=2, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Définir la grille des hyperparamètres
param_grid = {
 'penalty': ['l1', 'l2'],
 'C': [0.1, 1, 10],
 'solver': ['liblinear', 'saga'] # 'saga' supporte L1 et L2
}

# Utiliser StratifiedKFold pour les problèmes de classification avec potentiellement un déséquilibre de classes
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Créer l'objet GridSearchCV
grid_search = GridSearchCV(LogisticRegression(max_iter=10000), param_grid, cv=cv, scoring='f1')

# Lancer la recherche
grid_search.fit(X_train, y_train)

# Afficher les meilleurs hyperparamètres
print("Meilleurs hyperparamètres :", grid_search.best_params_)

# Obtenir le meilleur modèle
best_model = grid_search.best_estimator_

# Evaluer le modèle sur les données test
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))


## Recherche meilleur seuil de décision

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

X_train_val, X_val, y_train_val, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Calculer les probabilités prédites sur l'ensemble de validation
y_prob = best_model.predict_proba(X_val)[:, 1]

# Optimiser le seuil en utilisant le F1-score
best_threshold = 0
best_f1 = 0
for threshold in np.arange(0.1, 0.9, 0.05):
 y_pred = (y_prob > threshold).astype(int)
 f1 = f1_score(y_val, y_pred)
 if f1 > best_f1:
 best_f1 = f1
 best_threshold = threshold

print(f"Meilleur seuil : {best_threshold}")
print(f"Meilleur F1-score sur la validation: {best_f1}")

# Utiliser le modèle avec le seuil optimal sur les données de test
y_pred_test_optimized = (best_model.predict_proba(X_test)[:, 1] > best_threshold).astype(int)
print("Rapport de classification sur le test avec seuil optimisé")
print(classification_report(y_test, y_pred_test_optimized))